# GLM-Based Soay Sheep IBM

Fit parametric vital-rate models to the 1,000-row baseline sample, generate a short pooled IBM dataset, and construct the fitted GLM IPM.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from ipm_utils import calculate_ipm_metrics, mk_K_glm, reproduction_fn_glm, simulate_sheep_ibm

In [2]:
baseline = pd.read_csv("baseline_ibm_data_1000.csv")
growth_model = smf.ols("z1 ~ z", baseline.dropna(subset=["z1"])).fit()
survival_model = smf.glm("Surv ~ z", baseline, family=sm.families.Binomial()).fit()
reproduction_model = smf.glm("Repr ~ z", baseline.dropna(subset=["Repr"]), family=sm.families.Binomial()).fit()
recruitment_model = smf.glm("Recr ~ 1", baseline.dropna(subset=["Recr"]), family=sm.families.Binomial()).fit()
recruit_size_model = smf.ols("Rcsz ~ z", baseline.dropna(subset=["Rcsz"])).fit()

In [3]:
m_par_est = {"surv_int": survival_model.params["Intercept"], "surv_slope": survival_model.params["z"], "growth_int": growth_model.params["Intercept"], "growth_slope": growth_model.params["z"], "growth_noise": np.sqrt(growth_model.scale), "repr_int": reproduction_model.params["Intercept"], "repr_slope": reproduction_model.params["z"], "recr_int": recruitment_model.params["Intercept"], "rcsz_int": recruit_size_model.params["Intercept"], "rcsz_slope": recruit_size_model.params["z"], "rcsz_noise": np.sqrt(recruit_size_model.scale), "female_prob": 0.5}
pd.Series(m_par_est).round(4)

surv_int       -9.9702
surv_slope      3.8784
growth_int      1.4241
growth_slope    0.5521
growth_noise    0.0780
repr_int       -5.1057
repr_slope      1.8841
recr_int        1.8608
rcsz_int        0.4047
rcsz_slope      0.6943
rcsz_noise      0.1590
female_prob     0.5000
dtype: float64

In [4]:
rng = np.random.default_rng(10)
result = simulate_sheep_ibm(m_par_est, n_years=5, init_pop_size=250, rng=rng, max_population=5000, retain="pooled")
sim_data = result["data"]
if len(sim_data) > 1000:
    sim_data = sim_data.sample(1000, replace=False, random_state=10).reset_index(drop=True)
sim_data.to_csv("glm_ibm_dataset.csv", index=False)
print(f"Pooled Observations: {len(sim_data)}")
sim_data.describe().round(3)

Pooled Observations: 645


,z,Surv,z1,Repr,Sex,Recr,Rcsz,yr
count,645.000,645.000,405.000,405.000,240.000,107.000,97.000,645.000
mean,2.742,0.628,2.987,0.593,0.446,0.907,2.358,2.155
std,0.277,0.484,0.149,0.492,0.498,0.292,0.218,1.123
min,1.818,0.000,2.579,0.000,0.000,0.000,1.818,1.000
25%,2.555,0.000,2.890,0.000,0.000,1.000,2.229,1.000
50%,2.761,1.000,3.003,1.000,0.000,1.000,2.352,2.000
75%,2.967,1.000,3.100,1.000,1.000,1.000,2.492,3.000
max,3.278,1.000,3.293,1.000,1.000,1.000,2.848,4.000


In [6]:
L, U, n_mesh = 1.8, 3.3, 250
glm_ipm = mk_K_glm(n_mesh, m_par_est, L, U, correction=True)
mesh = glm_ipm["mesh_points"]
glm_metrics = calculate_ipm_metrics(glm_ipm["K"], reproduction_fn_glm(mesh, m_par_est))
np.savez_compressed("true_glm_metrics.npz", **glm_metrics)
print(f"GLM IBM Lambda: {glm_metrics['lambda']:.6f}")

GLM IBM Lambda: 1.006644
